# Reserve R1 baseline: a bean doctor in your pocket · **EDGE-CPU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a small CNN on 128 × 128 leaf photos (3 classes), measure test accuracy, and time it at batch 1 with ONNX Runtime on **1 thread**.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 1. Data

In [ ]:
from capstone.data.beans import load, CLASSES
X_tr, y_tr = load("train", max_items=200 if FAST else None); X_va, y_va = load("val"); X_te, y_te = load("test")
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape, "| classes", CLASSES, np.bincount(y_tr))
XTR, XVA, XTE = [X.astype(np.float32) / 255 for X in (X_tr, X_va, X_te)]

## 2. The model, costed before training

In [ ]:
model = nn.Sequential(nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU(),
                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 3))
p = profile(model, (3, 128, 128), bytes_per_weight=4)          # fp32 file
print(f"params {p['params']:,} | MACs {p['macs']:,} | fp32 weights {p['weight_bytes']:,} B")

## 3. Train (with random flips) and measure

In [ ]:
XTR_aug = np.concatenate([XTR, XTR[..., ::-1], XTR[..., ::-1, :]]).copy(); y_aug = np.concatenate([y_tr] * 3)
fit(model, XTR_aug, y_aug, epochs=1 if FAST else 15, bs=32, lr=2e-3)
ACC = np.mean(predict(model, XTE).argmax(1) == y_te)
print(f"test accuracy: {ACC:.4f}   (val: {np.mean(predict(model, XVA).argmax(1) == y_va):.4f})")

## 4. Latency at batch 1, ONNX Runtime, 1 thread
Measured here, then multiplied by the slow-down factor *k* on your chit card for the phone-class CPU.

In [ ]:
to_onnx(model, XTR[:1], "my_models/beans_fp32.onnx")
sess = ort_session("my_models/beans_fp32.onnx", threads=1)
LAT = ort_latency(sess, XTE[:1])
SIZE_MB = os.path.getsize("my_models/beans_fp32.onnx") / 1e6
print(f"batch 1, 1 thread: median {LAT['median_us'] / 1e3:.2f} ms, p99 {LAT['p99_us'] / 1e3:.2f} ms | model file {SIZE_MB:.2f} MB")

In [ ]:
ROW = report("R1 beans", (len(XTR), len(XVA), len(XTE)), (3, 128, 128), f"accuracy {ACC:.4f}", p,
             f"ORT 1 thread (this machine): median {LAT['median_us'] / 1e3:.2f} ms, p99 {LAT['p99_us'] / 1e3:.2f} ms; file {SIZE_MB:.2f} MB",
             "MIT (GitHub + HF, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.